# TP2 — Complexité et généralisation

**Master 2 MIAGE — Introduction au Machine Learning**

Dans le TP1, nous avons construit un premier modèle capable de prédire le délai réel de livraison d'une commande.

Nous allons maintenant nous poser une autre question :

> **Comment améliorer les performances d'un modèle ?**

Nous conservons la même cible, `delivery_time_days`, et la même séparation entre entraînement, validation et test.

### Objectifs

À la fin du TP, vous devrez être capable de :

- faire varier la complexité d'un modèle ;
- comparer ses performances sur entraînement et validation ;
- identifier expérimentalement sous-apprentissage et sur-apprentissage ;
- comprendre l'intérêt de la régularisation ;
- étudier l'effet de Ridge et Lasso.

## 1. Reprendre notre problème de régression

Nous cherchons toujours à prédire le **délai réel de livraison** d'une commande Olist.

La variable cible $Y$ correspond à `delivery_time_days`.

Dans ce TP, nous travaillons volontairement avec un petit nombre de variables numériques afin de contrôler facilement la complexité du modèle.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "ndiayemairame/miage-machine-learning/"
    "main/data/orders_ml.csv"
)

date_columns = [
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_ml = pd.read_csv(DATA_URL, parse_dates=date_columns)

print("Dimensions :", orders_ml.shape)
orders_ml.head()

## 2. Retrouver le même découpage que dans le TP1

Nous utilisons le même `random_state` et les mêmes proportions que dans le TP1 afin de conserver un protocole comparable.

In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(
    orders_ml,
    test_size=0.30,
    random_state=42
)

validation, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=42
)

print("Train      :", train.shape)
print("Validation :", validation.shape)
print("Test       :", test.shape)

## 3. Une première manière d'augmenter la complexité

Une régression linéaire classique construit une relation linéaire entre les variables explicatives et la prédiction.

Nous pouvons cependant enrichir les variables fournies au modèle. À partir de deux variables `x1` et `x2`, nous pouvons par exemple ajouter leurs carrés ou leur produit.

Le modèle reste une régression linéaire par rapport à ses coefficients, mais il dispose désormais d'une représentation plus riche des données.

Dans `scikit-learn`, `PolynomialFeatures` permet de réaliser automatiquement cette transformation.

Pour rendre l'expérience lisible et limiter le nombre de variables générées, nous utiliserons quatre variables numériques :

- `promised_delivery_days`
- `total_freight`
- `seller_customer_distance_mean`
- `mean_product_weight_g`

Pour cette expérience, nous retirons les commandes pour lesquelles l'une de ces informations est manquante.

In [ ]:
features = [
    "promised_delivery_days",
    "total_freight",
    "seller_customer_distance_mean",
    "mean_product_weight_g",
]

target = "delivery_time_days"

train_poly = train.dropna(subset=features).copy()
validation_poly = validation.dropna(subset=features).copy()

X_train = train_poly[features]
y_train = train_poly[target]

X_val = validation_poly[features]
y_val = validation_poly[target]

print("Train exploitable      :", X_train.shape)
print("Validation exploitable :", X_val.shape)

### Une nouvelle transformation

Observons concrètement ce que produit un polynôme de degré 2 sur nos quatre variables.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_demo = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_demo = poly_demo.fit_transform(X_train)

print("Nombre de variables avant :", X_train.shape[1])
print("Nombre de variables après :", X_demo.shape[1])

poly_demo.get_feature_names_out(features)

### Analyse

**Q4.** Combien de variables avions-nous avant la transformation ? Combien en avons-nous après ?

**Q5.** Concrètement, qu'a fait la transformation ?

**Q6.** Pourquoi peut-on dire que le modèle devient plus flexible alors que l'algorithme utilisé ensuite sera toujours `LinearRegression` ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 4. Mesurer la performance sur train et validation

Cette fois, nous voulons comprendre ce qui se passe lorsque la complexité augmente. Nous allons donc observer simultanément l'erreur sur les données d'entraînement et l'erreur sur des données non utilisées pour apprendre les coefficients.

Nous utiliserons la RMSE pour suivre cette évolution.

In [ ]:
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## 5. Modèle de degré 1

Avec `degree=1`, `PolynomialFeatures` ne crée pas de carré ni d'interaction : nous retrouvons essentiellement nos variables d'origine.

Le code du premier modèle est fourni afin de fixer le protocole que nous réutiliserons ensuite.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

degree_1_model = Pipeline([
    ("poly", PolynomialFeatures(degree=1, include_bias=False)),
    ("scaler", StandardScaler()),
    ("regression", LinearRegression())
])

degree_1_model.fit(X_train, y_train)

pred_train_d1 = degree_1_model.predict(X_train)
pred_val_d1 = degree_1_model.predict(X_val)

rmse_train_d1 = rmse(y_train, pred_train_d1)
rmse_val_d1 = rmse(y_val, pred_val_d1)

pd.DataFrame({
    "RMSE train": [rmse_train_d1],
    "RMSE validation": [rmse_val_d1]
}, index=["Degré 1"])

### Première observation

**Q7.** Comparez la RMSE d'entraînement et la RMSE de validation.

**Q8.** Pourquoi la performance sur les données d'entraînement ne suffit-elle pas pour juger de la qualité d'un modèle ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 6. Augmenter le degré

Le protocole ne change pas : seule la valeur du degré de `PolynomialFeatures` est modifiée.

### À vous de jouer

Construisez et évaluez un modèle de **degré 2** en vous inspirant directement du modèle de degré 1.

In [ ]:
# À VOUS DE JOUER

# degree_2_model = ...

# degree_2_model.fit(...)

# pred_train_d2 = ...
# pred_val_d2 = ...

# rmse_train_d2 = ...
# rmse_val_d2 = ...

Construisez maintenant de la même manière un modèle de **degré 3**.

In [ ]:
# À VOUS DE JOUER

# degree_3_model = ...

# degree_3_model.fit(...)

# pred_train_d3 = ...
# pred_val_d3 = ...

# rmse_train_d3 = ...
# rmse_val_d3 = ...

Rassemblons les résultats obtenus.

In [ ]:
complexity_results = pd.DataFrame({
    "degre": [1, 2, 3],
    "RMSE_train": [rmse_train_d1, rmse_train_d2, rmse_train_d3],
    "RMSE_validation": [rmse_val_d1, rmse_val_d2, rmse_val_d3],
})

complexity_results

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(complexity_results["degre"], complexity_results["RMSE_train"],
         marker="o", label="Train")
plt.plot(complexity_results["degre"], complexity_results["RMSE_validation"],
         marker="o", label="Validation")

plt.xlabel("Degré du polynôme")
plt.ylabel("RMSE")
plt.xticks(complexity_results["degre"])
plt.legend()
plt.show()

### Interpréter la complexité

**Q9.** Comment évolue l'erreur d'entraînement lorsque le degré augmente ?

**Q10.** L'erreur de validation évolue-t-elle nécessairement de la même manière ?

**Q11.** Quel degré choisiriez-vous à ce stade ? Justifiez uniquement à partir de l'expérience réalisée.

**Q12.** Un modèle qui obtient une erreur d'entraînement plus faible est-il nécessairement un meilleur modèle pour de nouvelles commandes ?

**Q13.** À partir de vos observations, expliquez avec vos propres mots ce que peuvent signifier **sous-apprentissage** et **sur-apprentissage**.

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 7. De la complexité au compromis biais-variance

L'expérience précédente met en évidence deux risques opposés.

Un modèle trop simple dispose de peu de liberté pour représenter la relation entre les variables et la cible. À l'inverse, un modèle très flexible peut devenir fortement dépendant des particularités des données utilisées pour l'entraîner.

On parle alors de **compromis biais-variance** :

- augmenter la complexité tend à réduire le biais ;
- mais peut augmenter la variance.

La difficulté n'est donc pas de construire le modèle le plus complexe possible, mais de trouver une complexité qui permette de bien **généraliser**.

### Réflexion

**Q14.** Associez les situations suivantes à un biais plutôt élevé ou une variance plutôt élevée :

1. le modèle est trop simple pour représenter la relation présente dans les données ;
2. le modèle change fortement lorsque l'on modifie légèrement les observations d'entraînement ;
3. le modèle reproduit très bien les données d'entraînement mais se dégrade sur validation.

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 8. Peut-on conserver un modèle riche tout en contrôlant sa complexité ?

Jusqu'ici, nous avons contrôlé la complexité en modifiant le degré du polynôme.

Une autre stratégie consiste à conserver une représentation riche, mais à **contraindre les coefficients** appris par le modèle.

C'est le principe de la **régularisation**.

Nous allons repartir d'une représentation polynomiale de degré 3 et utiliser une régression Ridge.

### Pourquoi standardiser ?

La régularisation agit sur la taille des coefficients.

Or nos variables n'ont pas les mêmes unités : jours, kilomètres, grammes, montant de transport, etc. Nous utilisons donc `StandardScaler` avant la régression afin de placer les variables transformées sur des échelles comparables.

## 9. Ridge : pénaliser les coefficients de grande amplitude

La régression Ridge ajoute une pénalisation de type L2.

Dans `scikit-learn`, l'intensité de cette régularisation est contrôlée par le paramètre `alpha`.

Commençons par une valeur intermédiaire.

In [ ]:
from sklearn.linear_model import Ridge

ridge_model = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

ridge_pred_train = ridge_model.predict(X_train)
ridge_pred_val = ridge_model.predict(X_val)

print("RMSE train      :", rmse(y_train, ridge_pred_train))
print("RMSE validation :", rmse(y_val, ridge_pred_val))

### Comprendre `alpha`

**Q15.** Quel est le rôle de `alpha` dans la régression Ridge ?

**Q16.** Que vous attendez-vous à observer si `alpha` devient très grand ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 10. Faire varier la régularisation

Nous allons maintenant entraîner plusieurs modèles Ridge en faisant varier uniquement `alpha`.

Complétez la liste avec plusieurs valeurs couvrant des intensités de régularisation très différentes, puis calculez pour chacune la RMSE sur train et validation.

In [ ]:
# À VOUS DE JOUER

#alphas = [
    # ...
#]

#ridge_results = []

#for alpha in alphas:

    # Pipeline : PolynomialFeatures degré 3
    # -> StandardScaler
    # -> Ridge(alpha=alpha)

    # model = ...
    # model.fit(...)

    # pred_train = ...
    # pred_val = ...

    # ridge_results.append({
    #     "alpha": alpha,
    #     "RMSE_train": ...,
    #     "RMSE_validation": ...
    # })

#ridge_results = pd.DataFrame(ridge_results)
#ridge_results

Une représentation graphique permet de mieux observer l'effet de `alpha`.

In [ ]:
plt.figure(figsize=(7, 4))

plt.semilogx(ridge_results["alpha"], ridge_results["RMSE_train"],
             marker="o", label="Train")
plt.semilogx(ridge_results["alpha"], ridge_results["RMSE_validation"],
             marker="o", label="Validation")

plt.xlabel("alpha")
plt.ylabel("RMSE")
plt.legend()
plt.show()

### Analyse

**Q17.** Comment évoluent les performances sur train lorsque `alpha` augmente ?

**Q18.** Quelle valeur de `alpha` donne la meilleure RMSE de validation parmi celles testées ?

**Q19.** Que se passe-t-il lorsque la régularisation devient trop forte ?

**Q20.** Expliquez pourquoi le meilleur `alpha` ne doit pas être choisi à partir de la performance sur le jeu de test.

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 11. Comment Ridge agit-il sur les coefficients ?

Comparons la taille des coefficients d'un modèle polynomial sans régularisation à celle d'un modèle Ridge.

Nous utilisons la même transformation polynomiale et la même standardisation afin que la comparaison soit cohérente.

In [ ]:
linear_degree_3 = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("scaler", StandardScaler()),
    ("regression", LinearRegression())
])

linear_degree_3.fit(X_train, y_train)

best_alpha = ridge_results.loc[
    ridge_results["RMSE_validation"].idxmin(),
    "alpha"
]

best_ridge = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=best_alpha))
])

best_ridge.fit(X_train, y_train)

coef_linear = linear_degree_3.named_steps["regression"].coef_
coef_ridge = best_ridge.named_steps["ridge"].coef_

coef_summary = pd.DataFrame({
    "Sans régularisation": coef_linear,
    "Ridge": coef_ridge
}).describe()

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    gridspec_kw={"width_ratios": [1, 1.5]}
)

axes[0].axis("off")

table = axes[0].table(
    cellText=coef_summary.round(3).values,
    rowLabels=coef_summary.index,
    colLabels=coef_summary.columns,
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.2)

axes[0].set_title("Résumé des coefficients")


axes[1].hist(
    coef_linear,
    bins=30,
    alpha=0.6,
    label="Sans régularisation"
)

axes[1].hist(
    coef_ridge,
    bins=30,
    alpha=0.6,
    label="Ridge"
)

axes[1].set_xlabel("Valeur des coefficients")
axes[1].set_ylabel("Nombre de coefficients")
axes[1].set_title("Distribution des coefficients")
axes[1].legend()

plt.tight_layout()
plt.show()

### Analyse

**Q21.** Comparez la dispersion et l'amplitude des coefficients avec et sans Ridge. Que constatez-vous ?

**Q22.** En quoi cette observation correspond-elle au principe de la régularisation étudié en cours ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 12. Lasso : réduire ou sélectionner ?

Ridge rapproche les coefficients de zéro sans chercher à les annuler.

La régression **Lasso** utilise une pénalisation différente, appelée L1, qui peut conduire certains coefficients à devenir exactement nuls.

Nous allons vérifier expérimentalement cette propriété sur la même représentation polynomiale.

In [ ]:
from sklearn.linear_model import Lasso

lasso_model = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("scaler", StandardScaler()),
    ("lasso", Lasso(alpha=0.01, max_iter=20000))
])

lasso_model.fit(X_train, y_train)

lasso_pred_val = lasso_model.predict(X_val)
lasso_coef = lasso_model.named_steps["lasso"].coef_

print("RMSE validation :", rmse(y_val, lasso_pred_val))
print("Nombre de coefficients :", len(lasso_coef))
print("Coefficients nuls :", np.sum(np.isclose(lasso_coef, 0)))

### Analyse

**Q23.** Certains coefficients sont-ils exactement nuls ?

**Q24.** Quelle différence observez-vous entre l'effet de Ridge et celui de Lasso sur les coefficients ?

**Q25.** Pourquoi peut-on dire que Lasso peut réaliser une forme de sélection automatique de variables ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 13. Choisir avant le test final
Nous avons maintenant réalisé plusieurs expériences :

- différents degrés polynomiaux ;
- différentes intensités de régularisation Ridge ;
- une première expérience avec Lasso.

À ce stade, le jeu de test n'a toujours pas été utilisé pour choisir le modèle.

### Décision

**Q26.** Parmi les modèles que vous avez comparés, lequel retiendriez-vous sur la base de la validation ? Justifiez à partir des résultats obtenus.

**Q27.** Quels hyperparamètres avez-vous choisis en utilisant le jeu de validation ?

**Q28.** Une fois ce choix effectué, pourquoi est-il pertinent de réentraîner le modèle sur `train + validation` avant l'évaluation finale ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 14. Évaluation finale

Une fois le modèle et ses hyperparamètres choisis, nous pouvons enfin utiliser le jeu de test.

Reprenez **le modèle que vous avez retenu**, réentraînez-le sur l'ensemble `train + validation`, puis évaluez-le une seule fois sur `test`.

Le code dépend de votre choix précédent : cette dernière étape est donc volontairement laissée à votre charge.

In [ ]:
# À VOUS DE JOUER

# 1. Regroupez train et validation.
# train_validation = ...

# 2. Construisez X et y avec les mêmes variables.
# ...

# 3. Reconstruisez le modèle retenu avec les hyperparamètres choisis.
# final_model = ...

# 4. Entraînez-le sur train + validation.
# final_model.fit(...)

# 5. Évaluez-le sur test.
# ...

### Dernière analyse

**Q29.** La performance sur test est-elle proche de celle observée sur validation ?

**Q30.** Si la performance test était sensiblement moins bonne que la performance validation, quelles explications pourriez-vous envisager ?

**Q31.** Pourquoi la performance d'entraînement seule aurait-elle été insuffisante pour sélectionner notre modèle ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 15. Bilan

Dans ce TP, nous n'avons pas changé de problème métier : nous avons toujours cherché à prédire le délai réel de livraison.

En revanche, nous avons modifié la **complexité de la famille de modèles** utilisée pour résoudre ce problème.

Retenez la chaîne de raisonnement suivante :

**complexifier → comparer train et validation → observer la généralisation → régulariser → choisir sur validation → évaluer une seule fois sur test**